# Statistical Analysis

## Data

### 1. Configure the analysis and helper functions


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from scipy.stats import rankdata, t, ttest_rel, wilcoxon

OUTPUT_ROOT = Path("Outputs")
MODELS = {
    "GPT-5.4 mini": "azure-gpt-5.4-mini",
    "GPT-OSS:20B": "gpt-oss:20b",
    "Gemma4:26B": "gemma4:26b",
    "Mixtral:8x22B": "mixtral-8x22b",
}
OUTCOMES = {
    "Process name": "processNames",
    "Analytic narrative": "analyticNarratives",
}
CONTRASTS = [
    {
        "Contrast": "Full vs actionable",
        "Baseline type": "full_set",
        "Comparison type": "reduced_set",
        "Baseline": "Full set",
        "Comparison": "Pharmacologically actionable",
    },
    {
        "Contrast": "Actionable vs noise 20%",
        "Baseline type": "reduced_set",
        "Comparison type": "noise_20",
        "Baseline": "Pharmacologically actionable",
        "Comparison": "Noise 20%",
    },
    {
        "Contrast": "Actionable vs noise 40%",
        "Baseline type": "reduced_set",
        "Comparison type": "noise_40",
        "Baseline": "Pharmacologically actionable",
        "Comparison": "Noise 40%",
    },
    {
        "Contrast": "Actionable vs noise 60%",
        "Baseline type": "reduced_set",
        "Comparison type": "noise_60",
        "Baseline": "Pharmacologically actionable",
        "Comparison": "Noise 60%",
    },
    {
        "Contrast": "Actionable vs noise 80%",
        "Baseline type": "reduced_set",
        "Comparison type": "noise_80",
        "Baseline": "Pharmacologically actionable",
        "Comparison": "Noise 80%",
    },
]
EXPECTED_PREDICTION_TYPES = {
    "full_set",
    "reduced_set",
    "noise_20",
    "noise_40",
    "noise_60",
    "noise_80",
}
SEMENTIC_SIMILARITY_BINS = [
    "Near identical",
    "Strong conceptual similarity",
    "Partial overlap",
    "Low / no semantic relevance",
]
SEMENTIC_SIMILARITY_BIN_CODES = {
    "Low / no semantic relevance":    1,
    "Partial overlap":                2,
    "Strong conceptual similarity":   3,
    "Near identical":                 4,
}
EXPECTED_PATHWAYS = 120
BOOTSTRAP_RESAMPLES = 20_000
BOOTSTRAP_SEED = 20260724

pd.set_option("display.max_rows", 100)
pd.set_option("display.max_columns", 30)

def bin_semantic(score: float) -> str:
    if score >= 0.9:
        return SEMENTIC_SIMILARITY_BINS[0]
    elif score >= 0.7:
        return SEMENTIC_SIMILARITY_BINS[1]
    elif score >= 0.5:
        return SEMENTIC_SIMILARITY_BINS[2]
    else:
        return SEMENTIC_SIMILARITY_BINS[3]

def holm_adjust(p_values):
    """Direct Holm step-down adjustment that preserves input order."""
    p_values = np.asarray(p_values, dtype=float)
    order = np.argsort(p_values)
    adjusted_sorted = np.maximum.accumulate(
        (len(p_values) - np.arange(len(p_values))) * p_values[order]
    )
    adjusted = np.empty(len(p_values), dtype=float)
    adjusted[order] = np.minimum(adjusted_sorted, 1.0)
    return adjusted


def mean_t_interval(values):
    """Arithmetic mean and two-sided 95% Student-t confidence interval."""
    values = np.asarray(values, dtype=float)
    mean = values.mean()
    half_width = t.ppf(0.975, len(values) - 1) * values.std(ddof=1) / np.sqrt(len(values))
    return mean, mean - half_width, mean + half_width


def mean_difference_interval(differences):
    """Mean paired difference and two-sided 95% Student-t confidence interval."""
    return mean_t_interval(differences)


def paired_cohens_dz(differences):
    """Paired Cohen's dz: mean difference divided by its sample standard deviation."""
    differences = np.asarray(differences, dtype=float)
    standard_deviation = differences.std(ddof=1)
    return differences.mean() / standard_deviation if standard_deviation > 0 else np.nan


def rank_biserial_from_differences(differences):
    """Matched-pairs rank-biserial correlation using signed rank sums."""
    nonzero = np.asarray(differences, dtype=float)
    nonzero = nonzero[nonzero != 0]
    if len(nonzero) == 0:
        return np.nan
    ranks = rankdata(np.abs(nonzero), method="average")
    positive_rank_sum = ranks[nonzero > 0].sum()
    negative_rank_sum = ranks[nonzero < 0].sum()
    return (positive_rank_sum - negative_rank_sum) / (positive_rank_sum + negative_rank_sum)


def median_iqr(values):
    """Median and interquartile range for an ordinal score."""
    values = np.asarray(values, dtype=float)
    median = np.median(values)
    q1, q3 = np.quantile(values, [0.25, 0.75])
    return median, q1, q3


### 2. Load and validate all model outputs

Every source file must contain 720 rows (120 pathways x 6 conditions), exactly one row per pathway--condition pair, complete numeric semantic scores, judge scores restricted to the ordered categories 1--4, and identical Reactome references across conditions.


In [2]:
frames = {}
validation_rows = []
required_columns = {
    "pathway_id",
    "reference",
    "prediction_type",
    "semantic_similarity",
    "llm_judge_score",
}

for model_label, model_directory in MODELS.items():
    for outcome_label, file_suffix in OUTCOMES.items():
        source_path = OUTPUT_ROOT / model_directory / f"evaluation_results_{file_suffix}.csv"
        frame = pd.read_csv(source_path)

        assert required_columns.issubset(frame.columns), f"Missing columns in {source_path}"
        assert len(frame) == EXPECTED_PATHWAYS * len(EXPECTED_PREDICTION_TYPES)
        assert set(frame["prediction_type"]) == EXPECTED_PREDICTION_TYPES
        assert not frame.duplicated(["pathway_id", "prediction_type"]).any()
        assert frame[["semantic_similarity", "llm_judge_score"]].notna().all().all()
        assert pd.api.types.is_numeric_dtype(frame["semantic_similarity"])
        assert np.isfinite(frame["semantic_similarity"]).all()
        assert set(frame["llm_judge_score"].unique()).issubset({1, 2, 3, 4})

        counts = frame.groupby("prediction_type")["pathway_id"].nunique()
        assert counts.eq(EXPECTED_PATHWAYS).all()

        reference_counts = frame.groupby("pathway_id")["reference"].nunique()
        assert reference_counts.eq(1).all()

        frame["semantic_bin"] = frame["semantic_similarity"].apply(bin_semantic).map(SEMENTIC_SIMILARITY_BIN_CODES)

        frames[(model_label, outcome_label)] = frame
        validation_rows.append(
            {
                "Model": model_label,
                "Output": outcome_label,
                "Rows": len(frame),
                "Pathways per condition": counts.min(),
                "Conditions": len(counts),
                "Unique semantic values": frame["semantic_similarity"].nunique(),
                "Judge categories": ", ".join(
                    str(value) for value in sorted(frame["llm_judge_score"].unique())
                ),
                "Duplicate pathway-condition rows": int(
                    frame.duplicated(["pathway_id", "prediction_type"]).sum()
                ),
                "Missing metric values": int(
                    frame[["semantic_similarity", "llm_judge_score"]].isna().sum().sum()
                ),
            }
        )

validation_summary = pd.DataFrame(validation_rows)
validation_summary


,Model,Output,Rows,Pathways per condition,Conditions,Unique semantic values,Judge categories,Duplicate pathway-condition rows,Missing metric values
0,GPT-5.4 mini,Process name,720,120,6,650,"1, 2, 3, 4",0,0
1,GPT-5.4 mini,Analytic narrative,720,120,6,720,"1, 2, 3, 4",0,0
2,GPT-OSS:20B,Process name,720,120,6,674,"1, 2, 3, 4",0,0
3,GPT-OSS:20B,Analytic narrative,720,120,6,720,"1, 2, 3, 4",0,0
4,Gemma4:26B,Process name,720,120,6,618,"1, 2, 3, 4",0,0
5,Gemma4:26B,Analytic narrative,720,120,6,720,"1, 2, 3, 4",0,0
6,Mixtral:8x22B,Process name,720,120,6,673,"1, 2, 3, 4",0,0
7,Mixtral:8x22B,Analytic narrative,720,120,6,720,"1, 2, 3",0,0


## Results

### 3. Run metric-appropriate paired tests

Continuous and ordinal outcomes follow separate analysis paths below. Raw paired-test results are calculated first, after which Holm adjustment is applied to the declared family for each experimental question: four LLMs for the full-versus-actionable analysis and four noise levels for the actionable-versus-noise analysis.

In [3]:
rng = np.random.default_rng(BOOTSTRAP_SEED)
continuous_rows = []
ordinal_rows = []

for model_label in MODELS:
    for outcome_label in OUTCOMES:
        frame = frames[(model_label, outcome_label)]

        for contrast in CONTRASTS:
            baseline = frame.loc[
                frame["prediction_type"] == contrast["Baseline type"],
                ["pathway_id", "reference", "semantic_similarity", "llm_judge_score"],
            ].rename(
                columns={
                    "reference": "reference_baseline",
                    "semantic_similarity": "semantic_baseline",
                    "llm_judge_score": "judge_baseline",
                }
            )
            comparison = frame.loc[
                frame["prediction_type"] == contrast["Comparison type"],
                ["pathway_id", "reference", "semantic_similarity", "llm_judge_score"],
            ].rename(
                columns={
                    "reference": "reference_comparison",
                    "semantic_similarity": "semantic_comparison",
                    "llm_judge_score": "judge_comparison",
                }
            )

            paired = baseline.merge(comparison, on="pathway_id", validate="one_to_one")
            assert len(paired) == EXPECTED_PATHWAYS
            assert (paired["reference_baseline"] == paired["reference_comparison"]).all()

            # Continuous semantic-similarity analysis: paired t-test on mean differences.
            semantic_differences = (
                paired["semantic_comparison"] - paired["semantic_baseline"]
            ).to_numpy()
            semantic_test = ttest_rel(
                paired["semantic_comparison"],
                paired["semantic_baseline"],
                alternative="two-sided",
            )
            baseline_mean, baseline_mean_ci_low, baseline_mean_ci_high = mean_t_interval(
                paired["semantic_baseline"]
            )
            comparison_mean, comparison_mean_ci_low, comparison_mean_ci_high = mean_t_interval(
                paired["semantic_comparison"]
            )
            mean_difference, mean_difference_ci_low, mean_difference_ci_high = (
                mean_difference_interval(semantic_differences)
            )
            continuous_rows.append(
                {
                    "Model": model_label,
                    "Output": outcome_label,
                    "Metric": "MedCPT semantic similarity",
                    "Scale": "Continuous",
                    "Test": "Paired t-test",
                    "Contrast": contrast["Contrast"],
                    "Baseline": contrast["Baseline"],
                    "Comparison": contrast["Comparison"],
                    "n": len(paired),
                    "Baseline mean": baseline_mean,
                    "Baseline mean CI low": baseline_mean_ci_low,
                    "Baseline mean CI high": baseline_mean_ci_high,
                    "Comparison mean": comparison_mean,
                    "Comparison mean CI low": comparison_mean_ci_low,
                    "Comparison mean CI high": comparison_mean_ci_high,
                    "Mean difference": mean_difference,
                    "Mean difference CI low": mean_difference_ci_low,
                    "Mean difference CI high": mean_difference_ci_high,
                    "t": semantic_test.statistic,
                    "df": len(paired) - 1,
                    "Raw p": semantic_test.pvalue,
                    "Cohen dz": paired_cohens_dz(semantic_differences),
                }
            )

            # Ordinal judge-score analysis: Wilcoxon signed-rank test and ordinal summaries.
            judge_differences = (
                paired["judge_comparison"] - paired["judge_baseline"]
            ).to_numpy()
            judge_test = wilcoxon(
                judge_differences,
                alternative="two-sided",
                zero_method="wilcox",
                correction=False,
                method="auto",
            )
            bootstrap_indices = rng.integers(
                0,
                len(judge_differences),
                size=(BOOTSTRAP_RESAMPLES, len(judge_differences)),
            )
            bootstrap_medians = np.median(judge_differences[bootstrap_indices], axis=1)
            median_difference_ci_low, median_difference_ci_high = np.quantile(
                bootstrap_medians, [0.025, 0.975]
            )
            baseline_median, baseline_q1, baseline_q3 = median_iqr(paired["judge_baseline"])
            comparison_median, comparison_q1, comparison_q3 = median_iqr(
                paired["judge_comparison"]
            )
            ordinal_rows.append(
                {
                    "Model": model_label,
                    "Output": outcome_label,
                    "Metric": "LLM-judge score",
                    "Scale": "Ordinal (1--4)",
                    "Test": "Wilcoxon signed-rank",
                    "Contrast": contrast["Contrast"],
                    "Baseline": contrast["Baseline"],
                    "Comparison": contrast["Comparison"],
                    "n": len(paired),
                    "Baseline median": baseline_median,
                    "Baseline Q1": baseline_q1,
                    "Baseline Q3": baseline_q3,
                    "Comparison median": comparison_median,
                    "Comparison Q1": comparison_q1,
                    "Comparison Q3": comparison_q3,
                    "Median difference": np.median(judge_differences),
                    "Median difference CI low": median_difference_ci_low,
                    "Median difference CI high": median_difference_ci_high,
                    "W": judge_test.statistic,
                    "Raw p": judge_test.pvalue,
                    "Rank-biserial r": rank_biserial_from_differences(judge_differences),
                    "Lower": int((judge_differences < 0).sum()),
                    "Higher": int((judge_differences > 0).sum()),
                    "Tied": int((judge_differences == 0).sum()),
                }
            )

continuous_results = pd.DataFrame(continuous_rows)
ordinal_results = pd.DataFrame(ordinal_rows)


def apply_declared_holm_families(results):
    """Apply Holm adjustment to the family defined for each experimental question."""
    results = results.copy()
    results["Holm p"] = np.nan
    results["Holm family"] = ""

    # Restriction question: one full-versus-actionable test for each of four LLMs.
    for outcome_label in OUTCOMES:
        family_mask = (
            results["Output"].eq(outcome_label)
            & results["Contrast"].eq("Full vs actionable")
        )
        assert family_mask.sum() == len(MODELS)
        results.loc[family_mask, "Holm p"] = holm_adjust(
            results.loc[family_mask, "Raw p"].to_numpy()
        )
        results.loc[family_mask, "Holm family"] = (
            "Full vs actionable across 4 LLMs"
        )

    # Noise question: four noise levels tested within each model and output.
    for model_label in MODELS:
        for outcome_label in OUTCOMES:
            family_mask = (
                results["Model"].eq(model_label)
                & results["Output"].eq(outcome_label)
                & results["Contrast"].ne("Full vs actionable")
            )
            assert family_mask.sum() == len(CONTRASTS) - 1
            results.loc[family_mask, "Holm p"] = holm_adjust(
                results.loc[family_mask, "Raw p"].to_numpy()
            )
            results.loc[family_mask, "Holm family"] = (
                "Actionable vs 4 noise levels within model"
            )

    return results


continuous_results = apply_declared_holm_families(continuous_results)
ordinal_results = apply_declared_holm_families(ordinal_results)

assert len(continuous_results) == len(MODELS) * len(OUTCOMES) * len(CONTRASTS)
assert len(ordinal_results) == len(MODELS) * len(OUTCOMES) * len(CONTRASTS)
assert continuous_results["Scale"].eq("Continuous").all()
assert continuous_results["Test"].eq("Paired t-test").all()
assert ordinal_results["Scale"].eq("Ordinal (1--4)").all()
assert ordinal_results["Test"].eq("Wilcoxon signed-rank").all()
assert continuous_results["n"].eq(EXPECTED_PATHWAYS).all()
assert ordinal_results["n"].eq(EXPECTED_PATHWAYS).all()
assert continuous_results["Holm p"].notna().all()
assert ordinal_results["Holm p"].notna().all()
assert continuous_results["Holm family"].ne("").all()
assert ordinal_results["Holm family"].ne("").all()
assert (continuous_results["Holm p"] + 1e-15 >= continuous_results["Raw p"]).all()
assert (ordinal_results["Holm p"] + 1e-15 >= ordinal_results["Raw p"]).all()

{
    "continuous_comparisons": len(continuous_results),
    "ordinal_comparisons": len(ordinal_results),
    "total_comparisons": len(continuous_results) + len(ordinal_results),
    "restriction_family_size": len(MODELS),
    "noise_family_size": len(CONTRASTS) - 1,
}

{'continuous_comparisons': 40,
 'ordinal_comparisons': 40,
 'total_comparisons': 80,
 'restriction_family_size': 4,
 'noise_family_size': 4}

### 4. Full set versus pharmacologically actionable subset

This section evaluates one restriction contrast across four LLMs. Continuous and ordinal results are presented separately so that their summaries, test statistics, and effect sizes remain appropriate to their measurement scales. Holm adjustment is applied across the four LLM-specific tests separately within each output--metric family.

In [4]:
def format_p(value):
    return f"{value:.4g}"


continuous_full_actionable = continuous_results.loc[
    continuous_results["Contrast"] == "Full vs actionable"
].copy()
continuous_full_actionable_display = pd.DataFrame(
    {
        "Model": continuous_full_actionable["Model"],
        "Output": continuous_full_actionable["Output"],
        "Baseline mean [95% CI]": continuous_full_actionable.apply(
            lambda row: (
                f"{row['Baseline mean']:.3f} "
                f"[{row['Baseline mean CI low']:.3f}, {row['Baseline mean CI high']:.3f}]"
            ),
            axis=1,
        ),
        "Actionable mean [95% CI]": continuous_full_actionable.apply(
            lambda row: (
                f"{row['Comparison mean']:.3f} "
                f"[{row['Comparison mean CI low']:.3f}, {row['Comparison mean CI high']:.3f}]"
            ),
            axis=1,
        ),
        "Mean difference [95% CI]": continuous_full_actionable.apply(
            lambda row: (
                f"{row['Mean difference']:.4f} "
                f"[{row['Mean difference CI low']:.4f}, {row['Mean difference CI high']:.4f}]"
            ),
            axis=1,
        ),
        "t (df=119)": continuous_full_actionable["t"].map(lambda value: f"{value:.3f}"),
        "Raw p": continuous_full_actionable["Raw p"].map(format_p),
        "Holm p (4 LLMs)": continuous_full_actionable["Holm p"].map(format_p),
        "Cohen dz": continuous_full_actionable["Cohen dz"].map(lambda value: f"{value:.3f}"),
    }
).reset_index(drop=True)

ordinal_full_actionable = ordinal_results.loc[
    ordinal_results["Contrast"] == "Full vs actionable"
].copy()
ordinal_full_actionable_display = pd.DataFrame(
    {
        "Model": ordinal_full_actionable["Model"],
        "Output": ordinal_full_actionable["Output"],
        "Baseline median [IQR]": ordinal_full_actionable.apply(
            lambda row: f"{row['Baseline median']:.1f} [{row['Baseline Q1']:.1f}, {row['Baseline Q3']:.1f}]",
            axis=1,
        ),
        "Actionable median [IQR]": ordinal_full_actionable.apply(
            lambda row: (
                f"{row['Comparison median']:.1f} "
                f"[{row['Comparison Q1']:.1f}, {row['Comparison Q3']:.1f}]"
            ),
            axis=1,
        ),
        "Median difference [95% CI]": ordinal_full_actionable.apply(
            lambda row: (
                f"{row['Median difference']:.1f} "
                f"[{row['Median difference CI low']:.1f}, {row['Median difference CI high']:.1f}]"
            ),
            axis=1,
        ),
        "W": ordinal_full_actionable["W"].map(lambda value: f"{value:.1f}"),
        "Raw p": ordinal_full_actionable["Raw p"].map(format_p),
        "Holm p (4 LLMs)": ordinal_full_actionable["Holm p"].map(format_p),
        "Rank-biserial r": ordinal_full_actionable["Rank-biserial r"].map(
            lambda value: f"{value:.3f}"
        ),
        "Lower / higher / tied": ordinal_full_actionable.apply(
            lambda row: f"{row['Lower']} / {row['Higher']} / {row['Tied']}", axis=1
        ),
    }
).reset_index(drop=True)

print("Continuous MedCPT semantic similarity: paired t-tests")
display(continuous_full_actionable_display)
print("Ordinal LLM-judge scores: paired Wilcoxon signed-rank tests")
display(ordinal_full_actionable_display)


Continuous MedCPT semantic similarity: paired t-tests


,Model,Output,Baseline mean [95% CI],Actionable mean [95% CI],Mean difference [95% CI],t (df=119),Raw p,Holm p (4 LLMs),Cohen dz
0,GPT-5.4 mini,Process name,"0.697 [0.669, 0.725]","0.678 [0.651, 0.704]","-0.0195 [-0.0416, 0.0027]",-1.741,0.08424,0.2527,-0.159
1,GPT-5.4 mini,Analytic narrative,"0.863 [0.848, 0.878]","0.853 [0.839, 0.866]","-0.0103 [-0.0188, -0.0018]",-2.392,0.01835,0.05504,-0.218
2,GPT-OSS:20B,Process name,"0.668 [0.642, 0.695]","0.657 [0.631, 0.684]","-0.0111 [-0.0328, 0.0106]",-1.016,0.3115,0.3546,-0.093
3,GPT-OSS:20B,Analytic narrative,"0.850 [0.835, 0.865]","0.852 [0.839, 0.865]","0.0016 [-0.0085, 0.0117]",0.309,0.7578,1,0.028
4,Gemma4:26B,Process name,"0.690 [0.665, 0.715]","0.663 [0.638, 0.688]","-0.0274 [-0.0465, -0.0082]",-2.831,0.005445,0.02178,-0.258
5,Gemma4:26B,Analytic narrative,"0.844 [0.829, 0.859]","0.830 [0.817, 0.844]","-0.0136 [-0.0239, -0.0034]",-2.632,0.00962,0.03848,-0.240
6,Mixtral:8x22B,Process name,"0.658 [0.633, 0.682]","0.645 [0.620, 0.670]","-0.0128 [-0.0316, 0.0059]",-1.357,0.1773,0.3546,-0.124
7,Mixtral:8x22B,Analytic narrative,"0.835 [0.821, 0.849]","0.833 [0.820, 0.845]","-0.0022 [-0.0122, 0.0079]",-0.429,0.6687,1,-0.039


Ordinal LLM-judge scores: paired Wilcoxon signed-rank tests


,Model,Output,Baseline median [IQR],Actionable median [IQR],Median difference [95% CI],W,Raw p,Holm p (4 LLMs),Rank-biserial r,Lower / higher / tied
0,GPT-5.4 mini,Process name,"2.0 [2.0, 3.0]","2.0 [2.0, 3.0]","0.0 [0.0, 0.0]",409.0,0.01891,0.05672,-0.358,32 / 18 / 70
1,GPT-5.4 mini,Analytic narrative,"2.0 [2.0, 3.0]","2.0 [2.0, 2.0]","0.0 [0.0, 0.0]",228.0,0.006545,0.02618,-0.444,28 / 12 / 80
2,GPT-OSS:20B,Process name,"2.0 [2.0, 3.0]","2.0 [2.0, 3.0]","0.0 [0.0, 0.0]",505.0,0.6744,0.6744,-0.066,23 / 23 / 74
3,GPT-OSS:20B,Analytic narrative,"2.0 [2.0, 2.0]","2.0 [2.0, 2.0]","0.0 [0.0, 0.0]",241.0,0.01524,0.03047,-0.412,29 / 11 / 80
4,Gemma4:26B,Process name,"2.0 [2.0, 3.0]","2.0 [2.0, 2.0]","0.0 [0.0, 0.0]",190.0,0.001013,0.004052,-0.537,30 / 10 / 80
5,Gemma4:26B,Analytic narrative,"2.0 [2.0, 2.0]","2.0 [2.0, 2.0]","0.0 [0.0, 0.0]",75.0,0.008151,0.02618,-0.538,19 / 6 / 95
6,Mixtral:8x22B,Process name,"2.0 [2.0, 2.0]","2.0 [2.0, 2.0]","0.0 [0.0, 0.0]",296.0,0.2258,0.4516,-0.201,22 / 16 / 82
7,Mixtral:8x22B,Analytic narrative,"2.0 [2.0, 2.0]","2.0 [2.0, 2.0]","0.0 [0.0, 0.0]",162.0,0.4652,0.4652,-0.143,15 / 12 / 93


### 5. Actionable subset versus each noise level

These contrasts isolate the effect of injected noise by using the unperturbed actionable subset as the baseline for all four noise levels. Holm adjustment is applied across the four noise-level tests separately within each model--output--metric family.

In [5]:
continuous_noise = continuous_results.loc[
    continuous_results["Contrast"] != "Full vs actionable",
    [
        "Model",
        "Output",
        "Contrast",
        "Baseline mean",
        "Comparison mean",
        "Mean difference",
        "Mean difference CI low",
        "Mean difference CI high",
        "t",
        "df",
        "Raw p",
        "Holm p",
        "Cohen dz",
    ],
].copy()
for column in [
    "Baseline mean",
    "Comparison mean",
    "Mean difference",
    "Mean difference CI low",
    "Mean difference CI high",
]:
    continuous_noise[column] = continuous_noise[column].map(lambda value: f"{value:.4f}")
continuous_noise["t"] = continuous_noise["t"].map(lambda value: f"{value:.3f}")
continuous_noise["Raw p"] = continuous_noise["Raw p"].map(format_p)
continuous_noise["Holm p"] = continuous_noise["Holm p"].map(format_p)
continuous_noise["Cohen dz"] = continuous_noise["Cohen dz"].map(lambda value: f"{value:.3f}")

ordinal_noise = ordinal_results.loc[
    ordinal_results["Contrast"] != "Full vs actionable",
    [
        "Model",
        "Output",
        "Contrast",
        "Baseline median",
        "Baseline Q1",
        "Baseline Q3",
        "Comparison median",
        "Comparison Q1",
        "Comparison Q3",
        "Median difference",
        "W",
        "Raw p",
        "Holm p",
        "Rank-biserial r",
        "Lower",
        "Higher",
        "Tied",
    ],
].copy()
for column in [
    "Baseline median",
    "Baseline Q1",
    "Baseline Q3",
    "Comparison median",
    "Comparison Q1",
    "Comparison Q3",
    "Median difference",
]:
    ordinal_noise[column] = ordinal_noise[column].map(lambda value: f"{value:.1f}")
ordinal_noise["W"] = ordinal_noise["W"].map(lambda value: f"{value:.1f}")
ordinal_noise["Raw p"] = ordinal_noise["Raw p"].map(format_p)
ordinal_noise["Holm p"] = ordinal_noise["Holm p"].map(format_p)
ordinal_noise["Rank-biserial r"] = ordinal_noise["Rank-biserial r"].map(
    lambda value: f"{value:.3f}"
)

print("Continuous MedCPT semantic similarity: paired t-tests")
display(
    continuous_noise.rename(
        columns={"Holm p": "Holm p (4 noise levels)"}
    ).reset_index(drop=True)
)
print("Ordinal LLM-judge scores: paired Wilcoxon signed-rank tests")
display(
    ordinal_noise.rename(
        columns={"Holm p": "Holm p (4 noise levels)"}
    ).reset_index(drop=True)
)


Continuous MedCPT semantic similarity: paired t-tests


,Model,Output,Contrast,Baseline mean,Comparison mean,Mean difference,Mean difference CI low,Mean difference CI high,t,df,Raw p,Holm p (4 noise levels),Cohen dz
0,GPT-5.4 mini,Process name,Actionable vs noise 20%,0.6775,0.6544,-0.0231,-0.0467,0.0005,-1.934,119,0.05547,0.05547,-0.177
1,GPT-5.4 mini,Process name,Actionable vs noise 40%,0.6775,0.6437,-0.0338,-0.0592,-0.0084,-2.632,119,0.00962,0.01924,-0.240
2,GPT-5.4 mini,Process name,Actionable vs noise 60%,0.6775,0.5969,-0.0806,-0.1064,-0.0549,-6.199,119,8.48e-09,2.544e-08,-0.566
3,GPT-5.4 mini,Process name,Actionable vs noise 80%,0.6775,0.5242,-0.1534,-0.1824,-0.1243,-10.450,119,1.564e-18,6.255e-18,-0.954
4,GPT-5.4 mini,Analytic narrative,Actionable vs noise 20%,0.8526,0.8449,-0.0077,-0.0150,-0.0005,-2.113,119,0.03669,0.03669,-0.193
5,GPT-5.4 mini,Analytic narrative,Actionable vs noise 40%,0.8526,0.8384,-0.0142,-0.0221,-0.0063,-3.542,119,0.0005671,0.001134,-0.323
6,GPT-5.4 mini,Analytic narrative,Actionable vs noise 60%,0.8526,0.8220,-0.0306,-0.0411,-0.0202,-5.806,119,5.427e-08,1.628e-07,-0.530
7,GPT-5.4 mini,Analytic narrative,Actionable vs noise 80%,0.8526,0.7841,-0.0685,-0.0816,-0.0554,-10.363,119,2.526e-18,1.01e-17,-0.946
8,GPT-OSS:20B,Process name,Actionable vs noise 20%,0.6574,0.6402,-0.0172,-0.0403,0.0059,-1.471,119,0.144,0.144,-0.134
9,GPT-OSS:20B,Process name,Actionable vs noise 40%,0.6574,0.6258,-0.0316,-0.0578,-0.0054,-2.386,119,0.01862,0.03724,-0.218


Ordinal LLM-judge scores: paired Wilcoxon signed-rank tests


,Model,Output,Contrast,Baseline median,Baseline Q1,Baseline Q3,Comparison median,Comparison Q1,Comparison Q3,Median difference,W,Raw p,Holm p (4 noise levels),Rank-biserial r,Lower,Higher,Tied
0,GPT-5.4 mini,Process name,Actionable vs noise 20%,2.0,2.0,3.0,2.0,2.0,2.0,0.0,514.5,0.1803,0.3606,-0.193,29,21,70
1,GPT-5.4 mini,Process name,Actionable vs noise 40%,2.0,2.0,3.0,2.0,2.0,3.0,0.0,630.0,0.1973,0.3606,-0.182,32,23,65
2,GPT-5.4 mini,Process name,Actionable vs noise 60%,2.0,2.0,3.0,2.0,1.0,2.0,0.0,428.5,1.923e-05,5.77e-05,-0.575,48,15,57
3,GPT-5.4 mini,Process name,Actionable vs noise 80%,2.0,2.0,3.0,1.0,1.0,2.0,-1.0,75.0,2.668e-13,1.067e-12,-0.944,70,3,47
4,GPT-5.4 mini,Analytic narrative,Actionable vs noise 20%,2.0,2.0,2.0,2.0,2.0,2.0,0.0,119.0,0.0001964,0.0001964,-0.643,29,7,84
5,GPT-5.4 mini,Analytic narrative,Actionable vs noise 40%,2.0,2.0,2.0,2.0,1.0,2.0,0.0,143.5,5.773e-06,1.155e-05,-0.710,37,7,76
6,GPT-5.4 mini,Analytic narrative,Actionable vs noise 60%,2.0,2.0,2.0,2.0,1.0,2.0,0.0,122.5,2.093e-09,6.278e-09,-0.846,51,5,64
7,GPT-5.4 mini,Analytic narrative,Actionable vs noise 80%,2.0,2.0,2.0,1.0,1.0,2.0,-1.0,66.0,8.999e-16,3.6e-15,-0.960,79,2,39
8,GPT-OSS:20B,Process name,Actionable vs noise 20%,2.0,2.0,3.0,2.0,2.0,2.0,0.0,415.0,0.1421,0.1421,-0.232,29,17,74
9,GPT-OSS:20B,Process name,Actionable vs noise 40%,2.0,2.0,3.0,2.0,2.0,2.0,0.0,283.0,0.001352,0.002703,-0.498,34,13,73


### 6. Count significant model-level contrasts

Counts range from 0 to 4 and summarize how many models showed a Holm-adjusted two-sided result below 0.05 for each output, metric, and contrast. For full versus actionable, adjustment is across four LLMs; for each noise contrast, adjustment is across four noise levels within a model. The counts are descriptive and do not replace model-specific estimates.

In [6]:
significance_counts = pd.concat(
    [
        continuous_results[["Output", "Metric", "Contrast", "Holm p"]],
        ordinal_results[["Output", "Metric", "Contrast", "Holm p"]],
    ],
    ignore_index=True,
)
significance_counts = (
    significance_counts.assign(Significant=significance_counts["Holm p"] < 0.05)
    .groupby(["Metric", "Output", "Contrast"], sort=False)["Significant"]
    .sum()
    .rename("Significant models (of 4)")
    .reset_index()
)

significance_counts


,Metric,Output,Contrast,Significant models (of 4)
0,MedCPT semantic similarity,Process name,Full vs actionable,1
1,MedCPT semantic similarity,Process name,Actionable vs noise 20%,0
2,MedCPT semantic similarity,Process name,Actionable vs noise 40%,4
3,MedCPT semantic similarity,Process name,Actionable vs noise 60%,4
4,MedCPT semantic similarity,Process name,Actionable vs noise 80%,4
5,MedCPT semantic similarity,Analytic narrative,Full vs actionable,1
6,MedCPT semantic similarity,Analytic narrative,Actionable vs noise 20%,2
7,MedCPT semantic similarity,Analytic narrative,Actionable vs noise 40%,4
8,MedCPT semantic similarity,Analytic narrative,Actionable vs noise 60%,4
9,MedCPT semantic similarity,Analytic narrative,Actionable vs noise 80%,4


### 7. Open-weight models versus the proprietary LLM: non-inferiority testing

The contrasts above test whether restricting or corrupting a model's own input changes that same model's score; they say nothing about whether one model's scores are as good as another's. This section asks a directional question: is each open-weight model's score not meaningfully worse than GPT-5.4 mini's, within a pre-specified margin? This is evaluated with one one-sided paired **non-inferiority test** per open-weight model, condition, output, and metric -- there is no separate two-sided difference test here, since "is there a detectable gap" is not the question being asked.

Comparisons pair GPT-5.4 mini's per-pathway score against each open-weight model's score (GPT-OSS:20B, Gemma4:26B, Mixtral:8x22B) on the same `pathway_id`, computed separately for the full-set and pharmacologically-actionable input conditions and for each output type (3 open-weight models x 2 conditions x 2 outputs x 2 metrics = 24 one-sided non-inferiority tests). Holm correction is applied within each condition-output family of 3 open-weight models.

- **MedCPT semantic similarity non-inferiority:** the margin is 0.5 Cohen's dz below zero (a conventional moderate-effect bound), converted to raw similarity units using each comparison's own paired-difference standard deviation. The test is a one-sided paired *t*-test of H0: mean(open-weight - proprietary) <= -margin (open-weight is inferior) against H1: mean difference > -margin (open-weight is non-inferior); non-inferiority holds when the one-sided 95% confidence interval lower bound falls above -margin.
- **LLM-judge score non-inferiority:** because the outcome is ordinal, non-inferiority is assessed on the rank-biserial correlation instead of a mean, with the same one-sided logic and a -0.5 rank-biserial r margin. The one-sided p-value and the 95% CI lower bound come from a 20,000-resample paired bootstrap (seed 20260725) rather than a parametric test.
- Non-inferiority is declared only when the Holm-adjusted one-sided p-value is below 0.05. Failing to reach significance means the data cannot rule out a meaningful degradation at this margin -- it is not evidence that one exists.


In [7]:
OPEN_WEIGHT_MODELS = ["GPT-OSS:20B", "Gemma4:26B", "Mixtral:8x22B"]
PROPRIETARY_MODEL = "GPT-5.4 mini"
NONINFERIORITY_CONDITIONS = [
    {"Condition": "Full set", "Prediction type": "full_set"},
    {"Condition": "Pharmacologically actionable", "Prediction type": "reduced_set"},
]
DZ_MARGIN = 0.5
RANK_BISERIAL_MARGIN = 0.5
NONINFERIORITY_ALPHA = 0.05
NONINFERIORITY_BOOTSTRAP_RESAMPLES = 20_000
NONINFERIORITY_BOOTSTRAP_SEED = 20260725

def format_p(value):
    return f"{value:.4g}"


def noninferiority_paired_t(differences, dz_margin, alpha):
    """One-sided non-inferiority test for a paired continuous outcome.

    Tests whether the open-weight model's score is not worse than the
    proprietary model's score by more than a pre-specified margin. The
    margin is a standardized effect size (Cohen's dz) converted to raw
    units using this comparison's own paired-difference standard
    deviation. H0: mean difference <= -margin (inferior); H1: mean
    difference > -margin (non-inferior).
    """
    n = len(differences)
    df = n - 1
    mean_difference = differences.mean()
    sd_difference = differences.std(ddof=1)
    se_difference = sd_difference / np.sqrt(n)
    margin_raw = dz_margin * sd_difference

    t_stat = (mean_difference + margin_raw) / se_difference
    noninferiority_p = 1 - t.cdf(t_stat, df)
    ci_low = mean_difference - t.ppf(1 - alpha, df) * se_difference

    return {
        "n": n,
        "Mean difference": mean_difference,
        "Margin (raw)": margin_raw,
        "CI low (95%, one-sided)": ci_low,
        "Cohen dz": mean_difference / sd_difference if sd_difference > 0 else np.nan,
        "Non-inferiority t": t_stat,
        "Non-inferiority raw p": noninferiority_p,
    }


def noninferiority_bootstrap_rank_biserial(differences, r_margin, alpha, resamples, seed):
    """Bootstrap one-sided non-inferiority test for the ordinal rank-biserial effect."""
    rng = np.random.default_rng(seed)
    n = len(differences)
    resampled = differences[rng.integers(0, n, size=(resamples, n))]
    boot_r = np.array([rank_biserial_from_differences(row) for row in resampled])
    boot_r = boot_r[~np.isnan(boot_r)]

    noninferiority_p = float((boot_r <= -r_margin).mean())
    ci_low = float(np.quantile(boot_r, alpha))

    return {
        "n": n,
        "Rank-biserial r": rank_biserial_from_differences(differences),
        "Margin": r_margin,
        "CI low (95%, one-sided)": ci_low,
        "Non-inferiority raw p": noninferiority_p,
    }


continuous_noninferiority_rows = []
ordinal_noninferiority_rows = []

for condition in NONINFERIORITY_CONDITIONS:
    for outcome_label in OUTCOMES:
        proprietary_frame = frames[(PROPRIETARY_MODEL, outcome_label)]
        proprietary_slice = proprietary_frame.loc[
            proprietary_frame["prediction_type"] == condition["Prediction type"],
            ["pathway_id", "semantic_similarity", "llm_judge_score"],
        ].rename(
            columns={
                "semantic_similarity": "semantic_proprietary",
                "llm_judge_score": "judge_proprietary",
            }
        )

        for open_weight_model in OPEN_WEIGHT_MODELS:
            open_weight_frame = frames[(open_weight_model, outcome_label)]
            open_weight_slice = open_weight_frame.loc[
                open_weight_frame["prediction_type"] == condition["Prediction type"],
                ["pathway_id", "semantic_similarity", "llm_judge_score"],
            ].rename(
                columns={
                    "semantic_similarity": "semantic_open_weight",
                    "llm_judge_score": "judge_open_weight",
                }
            )

            paired = proprietary_slice.merge(
                open_weight_slice, on="pathway_id", validate="one_to_one"
            )
            assert len(paired) == EXPECTED_PATHWAYS

            # Continuous semantic-similarity: one-sided non-inferiority test.
            semantic_differences = (
                paired["semantic_open_weight"] - paired["semantic_proprietary"]
            ).to_numpy()
            continuous_noninferiority_rows.append(
                {
                    "Condition": condition["Condition"],
                    "Output": outcome_label,
                    "Open-weight model": open_weight_model,
                    **noninferiority_paired_t(
                        semantic_differences, DZ_MARGIN, NONINFERIORITY_ALPHA
                    ),
                }
            )

            # Ordinal judge-score: one-sided bootstrap non-inferiority test.
            judge_differences = (
                paired["judge_open_weight"] - paired["judge_proprietary"]
            ).to_numpy()
            ordinal_noninferiority_rows.append(
                {
                    "Condition": condition["Condition"],
                    "Output": outcome_label,
                    "Open-weight model": open_weight_model,
                    "Lower": int((judge_differences < 0).sum()),
                    "Higher": int((judge_differences > 0).sum()),
                    "Tied": int((judge_differences == 0).sum()),
                    **noninferiority_bootstrap_rank_biserial(
                        judge_differences,
                        RANK_BISERIAL_MARGIN,
                        NONINFERIORITY_ALPHA,
                        NONINFERIORITY_BOOTSTRAP_RESAMPLES,
                        NONINFERIORITY_BOOTSTRAP_SEED,
                    ),
                }
            )

continuous_noninferiority = pd.DataFrame(continuous_noninferiority_rows)
ordinal_noninferiority = pd.DataFrame(ordinal_noninferiority_rows)


def apply_noninferiority_holm(results):
    """Holm-adjust the non-inferiority p-values within each condition-output
    family of 3 open-weight models."""
    results = results.copy()
    results["Holm p"] = np.nan
    for condition_name in results["Condition"].unique():
        for outcome_label in OUTCOMES:
            family_mask = results["Condition"].eq(condition_name) & results["Output"].eq(
                outcome_label
            )
            assert family_mask.sum() == len(OPEN_WEIGHT_MODELS)
            results.loc[family_mask, "Holm p"] = holm_adjust(
                results.loc[family_mask, "Non-inferiority raw p"].to_numpy()
            )
    results["Non-inferior"] = results["Holm p"] < NONINFERIORITY_ALPHA
    return results


continuous_noninferiority = apply_noninferiority_holm(continuous_noninferiority)
ordinal_noninferiority = apply_noninferiority_holm(ordinal_noninferiority)

assert len(continuous_noninferiority) == len(NONINFERIORITY_CONDITIONS) * len(OUTCOMES) * len(
    OPEN_WEIGHT_MODELS
)
assert len(ordinal_noninferiority) == len(NONINFERIORITY_CONDITIONS) * len(OUTCOMES) * len(
    OPEN_WEIGHT_MODELS
)

{
    "continuous_comparisons": len(continuous_noninferiority),
    "ordinal_comparisons": len(ordinal_noninferiority),
    "noninferiority_tests": len(continuous_noninferiority) + len(ordinal_noninferiority),
}

{'continuous_comparisons': 12,
 'ordinal_comparisons': 12,
 'noninferiority_tests': 24}

In [8]:
continuous_noninferiority_display = continuous_noninferiority.copy()
continuous_noninferiority_display["Mean difference [>= 95% CI low, one-sided]"] = (
    continuous_noninferiority_display.apply(
        lambda row: f"{row['Mean difference']:.4f} [>= {row['CI low (95%, one-sided)']:.4f}]",
        axis=1,
    )
)
continuous_noninferiority_display["Margin (raw, one-sided)"] = continuous_noninferiority_display[
    "Margin (raw)"
].map(lambda value: f"-{value:.4f}")
continuous_noninferiority_display["Cohen dz"] = continuous_noninferiority_display["Cohen dz"].map(
    lambda value: f"{value:.3f}"
)
continuous_noninferiority_display["t (df=119)"] = continuous_noninferiority_display[
    "Non-inferiority t"
].map(lambda value: f"{value:.3f}")
continuous_noninferiority_display["Raw p"] = continuous_noninferiority_display[
    "Non-inferiority raw p"
].map(format_p)
continuous_noninferiority_display["Holm p (3 models)"] = continuous_noninferiority_display[
    "Holm p"
].map(format_p)
continuous_noninferiority_display = continuous_noninferiority_display[
    [
        "Condition",
        "Output",
        "Open-weight model",
        "n",
        "Mean difference [>= 95% CI low, one-sided]",
        "Margin (raw, one-sided)",
        "Cohen dz",
        "t (df=119)",
        "Raw p",
        "Holm p (3 models)",
        "Non-inferior",
    ]
].reset_index(drop=True)

ordinal_noninferiority_display = ordinal_noninferiority.copy()
ordinal_noninferiority_display["Rank-biserial r [>= 95% CI low, one-sided]"] = (
    ordinal_noninferiority_display.apply(
        lambda row: f"{row['Rank-biserial r']:.3f} [>= {row['CI low (95%, one-sided)']:.3f}]",
        axis=1,
    )
)
ordinal_noninferiority_display["Margin (one-sided)"] = ordinal_noninferiority_display[
    "Margin"
].map(lambda value: f"-{value:.2f}")
ordinal_noninferiority_display["Raw p"] = ordinal_noninferiority_display[
    "Non-inferiority raw p"
].map(format_p)
ordinal_noninferiority_display["Holm p (3 models)"] = ordinal_noninferiority_display["Holm p"].map(
    format_p
)
ordinal_noninferiority_display["Lower / higher / tied"] = ordinal_noninferiority_display.apply(
    lambda row: f"{row['Lower']} / {row['Higher']} / {row['Tied']}", axis=1
)
ordinal_noninferiority_display = ordinal_noninferiority_display[
    [
        "Condition",
        "Output",
        "Open-weight model",
        "n",
        "Rank-biserial r [>= 95% CI low, one-sided]",
        "Margin (one-sided)",
        "Raw p",
        "Holm p (3 models)",
        "Non-inferior",
        "Lower / higher / tied",
    ]
].reset_index(drop=True)

print("Continuous MedCPT semantic similarity: one-sided paired non-inferiority tests")
display(continuous_noninferiority_display[continuous_noninferiority_display["Output"] == "Process name"])
print("Ordinal LLM-judge scores: one-sided bootstrap non-inferiority tests")
display(ordinal_noninferiority_display[ordinal_noninferiority_display["Output"] == "Process name"])

Continuous MedCPT semantic similarity: one-sided paired non-inferiority tests


,Condition,Output,Open-weight model,n,"Mean difference [>= 95% CI low, one-sided]","Margin (raw, one-sided)",Cohen dz,t (df=119),Raw p,Holm p (3 models),Non-inferior
0,Full set,Process name,GPT-OSS:20B,120,-0.0285 [>= -0.0450],-0.0546,-0.260,2.624,0.004917,0.009834,True
1,Full set,Process name,Gemma4:26B,120,-0.0069 [>= -0.0227],-0.0519,-0.067,4.746,2.916e-06,8.748e-06,True
2,Full set,Process name,Mixtral:8x22B,120,-0.0391 [>= -0.0559],-0.0556,-0.352,1.625,0.05343,0.05343,False
6,Pharmacologically actionable,Process name,GPT-OSS:20B,120,-0.0201 [>= -0.0393],-0.0633,-0.159,3.734,0.0001454,0.0002908,True
7,Pharmacologically actionable,Process name,Gemma4:26B,120,-0.0148 [>= -0.0329],-0.0597,-0.124,4.115,3.585e-05,0.0001075,True
8,Pharmacologically actionable,Process name,Mixtral:8x22B,120,-0.0325 [>= -0.0523],-0.0655,-0.248,2.760,0.003344,0.003344,True


Ordinal LLM-judge scores: one-sided bootstrap non-inferiority tests


,Condition,Output,Open-weight model,n,"Rank-biserial r [>= 95% CI low, one-sided]",Margin (one-sided),Raw p,Holm p (3 models),Non-inferior,Lower / higher / tied
0,Full set,Process name,GPT-OSS:20B,120,-0.317 [>= -0.551],-0.50,0.1042,0.2085,False,29 / 17 / 74
1,Full set,Process name,Gemma4:26B,120,-0.171 [>= -0.416],-0.50,0.01225,0.03675,True,23 / 22 / 75
2,Full set,Process name,Mixtral:8x22B,120,-0.616 [>= -0.787],-0.50,0.8527,0.8527,False,37 / 11 / 72
6,Pharmacologically actionable,Process name,GPT-OSS:20B,120,0.019 [>= -0.250],-0.50,0.00045,0.00135,True,19 / 22 / 79
7,Pharmacologically actionable,Process name,Gemma4:26B,120,-0.184 [>= -0.421],-0.50,0.01425,0.0285,True,27 / 20 / 73
8,Pharmacologically actionable,Process name,Mixtral:8x22B,120,-0.406 [>= -0.624],-0.50,0.2545,0.2545,False,30 / 15 / 75


In [9]:
noninferiority_counts = pd.concat(
    [
        continuous_noninferiority.assign(Metric="MedCPT semantic similarity"),
        ordinal_noninferiority.assign(Metric="LLM-judge score"),
    ],
    ignore_index=True,
)
noninferior_counts = (
    noninferiority_counts.groupby(["Metric", "Condition", "Output"], sort=False)["Non-inferior"]
    .sum()
    .rename("Non-inferior open-weight models (of 3)")
    .reset_index()
)

print("One-sided non-inferiority test: count non-inferior open-weight models")
display(noninferior_counts)

One-sided non-inferiority test: count non-inferior open-weight models


,Metric,Condition,Output,Non-inferior open-weight models (of 3)
0,MedCPT semantic similarity,Full set,Process name,2
1,MedCPT semantic similarity,Full set,Analytic narrative,1
2,MedCPT semantic similarity,Pharmacologically actionable,Process name,3
3,MedCPT semantic similarity,Pharmacologically actionable,Analytic narrative,1
4,LLM-judge score,Full set,Process name,1
5,LLM-judge score,Full set,Analytic narrative,0
6,LLM-judge score,Pharmacologically actionable,Process name,2
7,LLM-judge score,Pharmacologically actionable,Analytic narrative,0


### 8. Open-weight models versus the proprietary LLM: superiority testing




In [10]:
OPEN_WEIGHT_MODELS = ["GPT-OSS:20B", "Gemma4:26B", "Mixtral:8x22B"]
PROPRIETARY_MODEL = "GPT-5.4 mini"
SUPERIORITY_CONDITIONS = [
    {"Condition": "Full set", "Prediction type": "full_set"},
    {"Condition": "Pharmacologically actionable", "Prediction type": "reduced_set"},
]
DZ_MARGIN = 0.5
RANK_BISERIAL_MARGIN = 0.5
SUPERIORITY_ALPHA = 0.05
SUPERIORITY_BOOTSTRAP_RESAMPLES = 20_000
SUPERIORITY_BOOTSTRAP_SEED = 20260725

def format_p(value):
    return f"{value:.4g}"


def superiority_paired_t(differences, dz_margin, alpha):
    """One-sided superiority test for a paired continuous outcome.

    Tests whether the open-weight model's score is significantly better than the
    proprietary model's score by more than a pre-specified margin. The
    margin is a standardized effect size (Cohen's dz) converted to raw
    units using this comparison's own paired-difference standard
    deviation. H0: mean difference <= margin (superior); H1: mean
    difference > margin (superior).
    """
    n = len(differences)
    df = n - 1
    mean_difference = differences.mean()
    sd_difference = differences.std(ddof=1)
    se_difference = sd_difference / np.sqrt(n)
    margin_raw = dz_margin * sd_difference

    t_stat = (mean_difference - margin_raw) / se_difference
    superiority_p = 1 - t.cdf(t_stat, df)
    ci_low = mean_difference - t.ppf(1 - alpha, df) * se_difference

    return {
        "n": n,
        "Mean difference": mean_difference,
        "Margin (raw)": margin_raw,
        "CI low (95%, one-sided)": ci_low,
        "Cohen dz": mean_difference / sd_difference if sd_difference > 0 else np.nan,
        "Superiority t": t_stat,
        "Superiority raw p": superiority_p,
    }


def superiority_bootstrap_rank_biserial(differences, r_margin, alpha, resamples, seed):
    """Bootstrap one-sided superiority test for the ordinal rank-biserial effect."""
    rng = np.random.default_rng(seed)
    n = len(differences)
    resampled = differences[rng.integers(0, n, size=(resamples, n))]
    boot_r = np.array([rank_biserial_from_differences(row) for row in resampled])
    boot_r = boot_r[~np.isnan(boot_r)]

    superiority_p = float((boot_r <= r_margin).mean())
    ci_low = float(np.quantile(boot_r, alpha))

    return {
        "n": n,
        "Rank-biserial r": rank_biserial_from_differences(differences),
        "Margin": r_margin,
        "CI low (95%, one-sided)": ci_low,
        "Superiority raw p": superiority_p,
    }


continuous_superiority_rows = []
ordinal_superiority_rows = []

for condition in SUPERIORITY_CONDITIONS:
    for outcome_label in OUTCOMES:
        proprietary_frame = frames[(PROPRIETARY_MODEL, outcome_label)]
        proprietary_slice = proprietary_frame.loc[
            proprietary_frame["prediction_type"] == condition["Prediction type"],
            ["pathway_id", "semantic_similarity", "llm_judge_score"],
        ].rename(
            columns={
                "semantic_similarity": "semantic_proprietary",
                "llm_judge_score": "judge_proprietary",
            }
        )

        for open_weight_model in OPEN_WEIGHT_MODELS:
            open_weight_frame = frames[(open_weight_model, outcome_label)]
            open_weight_slice = open_weight_frame.loc[
                open_weight_frame["prediction_type"] == condition["Prediction type"],
                ["pathway_id", "semantic_similarity", "llm_judge_score"],
            ].rename(
                columns={
                    "semantic_similarity": "semantic_open_weight",
                    "llm_judge_score": "judge_open_weight",
                }
            )

            paired = proprietary_slice.merge(
                open_weight_slice, on="pathway_id", validate="one_to_one"
            )
            assert len(paired) == EXPECTED_PATHWAYS

            # Continuous semantic-similarity: one-sided non-inferiority test.
            semantic_differences = (
                paired["semantic_open_weight"] - paired["semantic_proprietary"]
            ).to_numpy()
            continuous_superiority_rows.append(
                {
                    "Condition": condition["Condition"],
                    "Output": outcome_label,
                    "Open-weight model": open_weight_model,
                    **superiority_paired_t(
                        semantic_differences, DZ_MARGIN, SUPERIORITY_ALPHA
                    ),
                }
            )

            # Ordinal judge-score: one-sided bootstrap non-inferiority test.
            judge_differences = (
                paired["judge_open_weight"] - paired["judge_proprietary"]
            ).to_numpy()
            ordinal_superiority_rows.append(
                {
                    "Condition": condition["Condition"],
                    "Output": outcome_label,
                    "Open-weight model": open_weight_model,
                    "Lower": int((judge_differences < 0).sum()),
                    "Higher": int((judge_differences > 0).sum()),
                    "Tied": int((judge_differences == 0).sum()),
                    **superiority_bootstrap_rank_biserial(
                        judge_differences,
                        RANK_BISERIAL_MARGIN,
                        SUPERIORITY_ALPHA,
                        SUPERIORITY_BOOTSTRAP_RESAMPLES,
                        SUPERIORITY_BOOTSTRAP_SEED,
                    ),
                }
            )

continuous_superiority = pd.DataFrame(continuous_superiority_rows)
ordinal_superiority = pd.DataFrame(ordinal_superiority_rows)


def apply_superiority_holm(results):
    """Holm-adjust the superiority p-values within each condition-output
    family of 3 open-weight models."""
    results = results.copy()
    results["Holm p"] = np.nan
    for condition_name in results["Condition"].unique():
        for outcome_label in OUTCOMES:
            family_mask = results["Condition"].eq(condition_name) & results["Output"].eq(
                outcome_label
            )
            assert family_mask.sum() == len(OPEN_WEIGHT_MODELS)
            results.loc[family_mask, "Holm p"] = holm_adjust(
                results.loc[family_mask, "Superiority raw p"].to_numpy()
            )
    results["Superior"] = results["Holm p"] < SUPERIORITY_ALPHA
    return results


continuous_superiority = apply_superiority_holm(continuous_superiority)
ordinal_superiority = apply_superiority_holm(ordinal_superiority)

assert len(continuous_superiority) == len(SUPERIORITY_CONDITIONS) * len(OUTCOMES) * len(
    OPEN_WEIGHT_MODELS
)
assert len(ordinal_superiority) == len(SUPERIORITY_CONDITIONS) * len(OUTCOMES) * len(
    OPEN_WEIGHT_MODELS
)

{
    "continuous_comparisons": len(continuous_superiority),
    "ordinal_comparisons": len(ordinal_superiority),
    "superiority_tests": len(continuous_superiority) + len(ordinal_superiority),
}

{'continuous_comparisons': 12,
 'ordinal_comparisons': 12,
 'superiority_tests': 24}

In [11]:
continuous_superiority_display = continuous_superiority.copy()
continuous_superiority_display["Mean difference [>= 95% CI low, one-sided]"] = (
    continuous_superiority_display.apply(
        lambda row: f"{row['Mean difference']:.4f} [>= {row['CI low (95%, one-sided)']:.4f}]",
        axis=1,
    )
)
continuous_superiority_display["Margin (raw, one-sided)"] = continuous_superiority_display[
    "Margin (raw)"
].map(lambda value: f"-{value:.4f}")
continuous_superiority_display["Cohen dz"] = continuous_superiority_display["Cohen dz"].map(
    lambda value: f"{value:.3f}"
)
continuous_superiority_display["t (df=119)"] = continuous_superiority_display[
    "Superiority t"
].map(lambda value: f"{value:.3f}")
continuous_superiority_display["Raw p"] = continuous_superiority_display[
    "Superiority raw p"
].map(format_p)
continuous_superiority_display["Holm p (3 models)"] = continuous_superiority_display[
    "Holm p"
].map(format_p)
continuous_superiority_display = continuous_superiority_display[
    [
        "Condition",
        "Output",
        "Open-weight model",
        "n",
        "Mean difference [>= 95% CI low, one-sided]",
        "Margin (raw, one-sided)",
        "Cohen dz",
        "t (df=119)",
        "Raw p",
        "Holm p (3 models)",
        "Superior",
    ]
].reset_index(drop=True)

ordinal_superiority_display = ordinal_superiority.copy()
ordinal_superiority_display["Rank-biserial r [>= 95% CI low, one-sided]"] = (
    ordinal_superiority_display.apply(
        lambda row: f"{row['Rank-biserial r']:.3f} [>= {row['CI low (95%, one-sided)']:.3f}]",
        axis=1,
    )
)
ordinal_superiority_display["Margin (one-sided)"] = ordinal_superiority_display[
    "Margin"
].map(lambda value: f"-{value:.2f}")
ordinal_superiority_display["Raw p"] = ordinal_superiority_display[
    "Superiority raw p"
].map(format_p)
ordinal_superiority_display["Holm p (3 models)"] = ordinal_superiority_display["Holm p"].map(
    format_p
)
ordinal_superiority_display["Lower / higher / tied"] = ordinal_superiority_display.apply(
    lambda row: f"{row['Lower']} / {row['Higher']} / {row['Tied']}", axis=1
)
ordinal_superiority_display = ordinal_superiority_display[
    [
        "Condition",
        "Output",
        "Open-weight model",
        "n",
        "Rank-biserial r [>= 95% CI low, one-sided]",
        "Margin (one-sided)",
        "Raw p",
        "Holm p (3 models)",
        "Superior",
        "Lower / higher / tied",
    ]
].reset_index(drop=True)

print("Continuous MedCPT semantic similarity: one-sided paired non-inferiority tests")
display(continuous_superiority_display[continuous_superiority_display["Output"] == "Process name"])
print("Ordinal LLM-judge scores: one-sided bootstrap non-inferiority tests")
display(ordinal_superiority_display[ordinal_superiority_display["Output"] == "Process name"])

Continuous MedCPT semantic similarity: one-sided paired non-inferiority tests


,Condition,Output,Open-weight model,n,"Mean difference [>= 95% CI low, one-sided]","Margin (raw, one-sided)",Cohen dz,t (df=119),Raw p,Holm p (3 models),Superior
0,Full set,Process name,GPT-OSS:20B,120,-0.0285 [>= -0.0450],-0.0546,-0.260,-8.331,1,1,False
1,Full set,Process name,Gemma4:26B,120,-0.0069 [>= -0.0227],-0.0519,-0.067,-6.209,1,1,False
2,Full set,Process name,Mixtral:8x22B,120,-0.0391 [>= -0.0559],-0.0556,-0.352,-9.330,1,1,False
6,Pharmacologically actionable,Process name,GPT-OSS:20B,120,-0.0201 [>= -0.0393],-0.0633,-0.159,-7.221,1,1,False
7,Pharmacologically actionable,Process name,Gemma4:26B,120,-0.0148 [>= -0.0329],-0.0597,-0.124,-6.840,1,1,False
8,Pharmacologically actionable,Process name,Mixtral:8x22B,120,-0.0325 [>= -0.0523],-0.0655,-0.248,-8.194,1,1,False


Ordinal LLM-judge scores: one-sided bootstrap non-inferiority tests


,Condition,Output,Open-weight model,n,"Rank-biserial r [>= 95% CI low, one-sided]",Margin (one-sided),Raw p,Holm p (3 models),Superior,Lower / higher / tied
0,Full set,Process name,GPT-OSS:20B,120,-0.317 [>= -0.551],-0.50,1,1,False,29 / 17 / 74
1,Full set,Process name,Gemma4:26B,120,-0.171 [>= -0.416],-0.50,1,1,False,23 / 22 / 75
2,Full set,Process name,Mixtral:8x22B,120,-0.616 [>= -0.787],-0.50,1,1,False,37 / 11 / 72
6,Pharmacologically actionable,Process name,GPT-OSS:20B,120,0.019 [>= -0.250],-0.50,0.9981,1,False,19 / 22 / 79
7,Pharmacologically actionable,Process name,Gemma4:26B,120,-0.184 [>= -0.421],-0.50,1,1,False,27 / 20 / 73
8,Pharmacologically actionable,Process name,Mixtral:8x22B,120,-0.406 [>= -0.624],-0.50,1,1,False,30 / 15 / 75


In [12]:
s_counts = pd.concat(
    [
        continuous_superiority.assign(Metric="MedCPT semantic similarity"),
        ordinal_superiority.assign(Metric="LLM-judge score"),
    ],
    ignore_index=True,
)
superiority_counts = (
    s_counts.groupby(["Metric", "Condition", "Output"], sort=False)["Superior"]
    .sum()
    .rename("Non-inferior open-weight models (of 3)")
    .reset_index()
)

print("One-sided superiority test: count superior open-weight models")
display(superiority_counts)

One-sided superiority test: count superior open-weight models


,Metric,Condition,Output,Non-inferior open-weight models (of 3)
0,MedCPT semantic similarity,Full set,Process name,0
1,MedCPT semantic similarity,Full set,Analytic narrative,0
2,MedCPT semantic similarity,Pharmacologically actionable,Process name,0
3,MedCPT semantic similarity,Pharmacologically actionable,Analytic narrative,0
4,LLM-judge score,Full set,Process name,0
5,LLM-judge score,Full set,Analytic narrative,0
6,LLM-judge score,Pharmacologically actionable,Process name,0
7,LLM-judge score,Pharmacologically actionable,Analytic narrative,0


### 9. Open-weight models versus the proprietary LLM: equivalence testing (TOST)

The non-inferiority tests above establish only that many open-weight models are not detectably worse than GPT-5.4 mini by more than the pre-specified margin; a model could in principle score much higher than the proprietary model and still pass a non-inferiority test. This section applies the complementary two one-sided tests (TOST) equivalence procedure to the same 24 open-weight-versus-proprietary comparisons (3 open-weight models x 2 conditions x 2 outputs x 2 metrics). Equivalence is the stronger, two-sided claim: it requires that the open-weight model is neither detectably worse (the non-inferiority criterion above) nor detectably better than the proprietary model by more than the margin, whereas non-inferiority alone permits a real improvement above the margin to still pass. Holm correction is applied within each condition-output family of 3 open-weight models, separately from the non-inferiority Holm families above.

- **MedCPT semantic similarity equivalence:** the margin is +/-0.5 Cohen's dz (the same magnitude used for the non-inferiority margin, applied on both sides), converted to raw similarity units using each comparison's own paired-difference standard deviation. TOST is run as two one-sided paired t-tests; two one-sided tests at alpha=0.05 correspond to a 90% CI for the mean difference, so equivalence holds when that 90% CI falls entirely inside the margin.
- **LLM-judge score equivalence:** because the outcome is ordinal, equivalence is assessed on the rank-biserial correlation instead of a mean, with margin +/-0.5 rank-biserial r. The one-sided p-values and 90% CI come from a 20,000-resample paired bootstrap (seed 20260726).
- Equivalence is declared only when the Holm-adjusted p-value is below 0.05.


In [13]:
EQUIVALENCE_CONDITIONS = [
    {"Condition": "Full set", "Prediction type": "full_set"},
    {"Condition": "Pharmacologically actionable", "Prediction type": "reduced_set"},
]
EQUIVALENCE_DZ_MARGIN = 0.5
EQUIVALENCE_RANK_BISERIAL_MARGIN = 0.5
EQUIVALENCE_ALPHA = 0.05
EQUIVALENCE_BOOTSTRAP_RESAMPLES = 20_000
EQUIVALENCE_BOOTSTRAP_SEED = 20260726

def format_p(value):
    return f"{value:.4g}"


def tost_paired_t(differences, dz_margin, alpha):
    """TOST equivalence test for a paired continuous outcome.

    The margin is a standardized effect size (Cohen's dz) converted to raw
    units using this comparison's own paired-difference standard deviation.
    """
    n = len(differences)
    df = n - 1
    mean_difference = differences.mean()
    sd_difference = differences.std(ddof=1)
    se_difference = sd_difference / np.sqrt(n)
    margin_raw = dz_margin * sd_difference

    p_lower = 1 - t.cdf((mean_difference + margin_raw) / se_difference, df)
    p_upper = t.cdf((mean_difference - margin_raw) / se_difference, df)
    tost_p = max(p_lower, p_upper)

    bound = t.ppf(1 - alpha, df) * se_difference
    return {
        "n": n,
        "Mean difference": mean_difference,
        "Margin (raw)": margin_raw,
        "CI low (90%)": mean_difference - bound,
        "CI high (90%)": mean_difference + bound,
        "Cohen dz": mean_difference / sd_difference if sd_difference > 0 else np.nan,
        "TOST raw p": tost_p,
    }


def tost_bootstrap_rank_biserial(differences, r_margin, alpha, resamples, seed):
    """Bootstrap TOST-style equivalence test for the ordinal rank-biserial effect."""
    rng = np.random.default_rng(seed)
    n = len(differences)
    resampled = differences[rng.integers(0, n, size=(resamples, n))]
    boot_r = np.array([rank_biserial_from_differences(row) for row in resampled])
    boot_r = boot_r[~np.isnan(boot_r)]

    p_lower = float((boot_r <= -r_margin).mean())
    p_upper = float((boot_r >= r_margin).mean())
    ci_low, ci_high = np.quantile(boot_r, [alpha, 1 - alpha])
    return {
        "n": n,
        "Rank-biserial r": rank_biserial_from_differences(differences),
        "Margin": r_margin,
        "CI low (90%)": ci_low,
        "CI high (90%)": ci_high,
        "TOST raw p": max(p_lower, p_upper),
    }


continuous_equivalence_rows = []
ordinal_equivalence_rows = []

for condition in EQUIVALENCE_CONDITIONS:
    for outcome_label in OUTCOMES:
        proprietary_frame = frames[(PROPRIETARY_MODEL, outcome_label)]
        proprietary_slice = proprietary_frame.loc[
            proprietary_frame["prediction_type"] == condition["Prediction type"],
            ["pathway_id", "semantic_similarity", "llm_judge_score"],
        ].rename(
            columns={
                "semantic_similarity": "semantic_proprietary",
                "llm_judge_score": "judge_proprietary",
            }
        )

        for open_weight_model in OPEN_WEIGHT_MODELS:
            open_weight_frame = frames[(open_weight_model, outcome_label)]
            open_weight_slice = open_weight_frame.loc[
                open_weight_frame["prediction_type"] == condition["Prediction type"],
                ["pathway_id", "semantic_similarity", "llm_judge_score"],
            ].rename(
                columns={
                    "semantic_similarity": "semantic_open_weight",
                    "llm_judge_score": "judge_open_weight",
                }
            )

            paired = proprietary_slice.merge(
                open_weight_slice, on="pathway_id", validate="one_to_one"
            )
            assert len(paired) == EXPECTED_PATHWAYS

            # Continuous semantic-similarity: paired TOST equivalence test.
            semantic_differences = (
                paired["semantic_open_weight"] - paired["semantic_proprietary"]
            ).to_numpy()
            continuous_equivalence_rows.append(
                {
                    "Condition": condition["Condition"],
                    "Output": outcome_label,
                    "Open-weight model": open_weight_model,
                    **tost_paired_t(
                        semantic_differences, EQUIVALENCE_DZ_MARGIN, EQUIVALENCE_ALPHA
                    ),
                }
            )

            # Ordinal judge-score: bootstrap TOST equivalence test.
            judge_differences = (
                paired["judge_open_weight"] - paired["judge_proprietary"]
            ).to_numpy()
            ordinal_equivalence_rows.append(
                {
                    "Condition": condition["Condition"],
                    "Output": outcome_label,
                    "Open-weight model": open_weight_model,
                    "Lower": int((judge_differences < 0).sum()),
                    "Higher": int((judge_differences > 0).sum()),
                    "Tied": int((judge_differences == 0).sum()),
                    **tost_bootstrap_rank_biserial(
                        judge_differences,
                        EQUIVALENCE_RANK_BISERIAL_MARGIN,
                        EQUIVALENCE_ALPHA,
                        EQUIVALENCE_BOOTSTRAP_RESAMPLES,
                        EQUIVALENCE_BOOTSTRAP_SEED,
                    ),
                }
            )

continuous_equivalence = pd.DataFrame(continuous_equivalence_rows)
ordinal_equivalence = pd.DataFrame(ordinal_equivalence_rows)


def apply_equivalence_holm(results):
    """Holm-adjust the TOST p-values within each condition-output family of
    3 open-weight models."""
    results = results.copy()
    results["Holm p"] = np.nan
    for condition_name in results["Condition"].unique():
        for outcome_label in OUTCOMES:
            family_mask = results["Condition"].eq(condition_name) & results["Output"].eq(
                outcome_label
            )
            assert family_mask.sum() == len(OPEN_WEIGHT_MODELS)
            results.loc[family_mask, "Holm p"] = holm_adjust(
                results.loc[family_mask, "TOST raw p"].to_numpy()
            )
    results["Equivalent"] = results["Holm p"] < EQUIVALENCE_ALPHA
    return results


continuous_equivalence = apply_equivalence_holm(continuous_equivalence)
ordinal_equivalence = apply_equivalence_holm(ordinal_equivalence)

assert len(continuous_equivalence) == len(EQUIVALENCE_CONDITIONS) * len(OUTCOMES) * len(
    OPEN_WEIGHT_MODELS
)
assert len(ordinal_equivalence) == len(EQUIVALENCE_CONDITIONS) * len(OUTCOMES) * len(
    OPEN_WEIGHT_MODELS
)

{
    "continuous_comparisons": len(continuous_equivalence),
    "ordinal_comparisons": len(ordinal_equivalence),
    "equivalence_tests": len(continuous_equivalence) + len(ordinal_equivalence),
}

{'continuous_comparisons': 12,
 'ordinal_comparisons': 12,
 'equivalence_tests': 24}

In [14]:
continuous_equivalence_display = continuous_equivalence.copy()
continuous_equivalence_display["Mean difference [90% CI]"] = continuous_equivalence_display.apply(
    lambda row: f"{row['Mean difference']:.4f} [{row['CI low (90%)']:.4f}, {row['CI high (90%)']:.4f}]",
    axis=1,
)
continuous_equivalence_display["Margin (raw, +/-)"] = continuous_equivalence_display[
    "Margin (raw)"
].map(lambda value: f"{value:.4f}")
continuous_equivalence_display["Cohen dz"] = continuous_equivalence_display["Cohen dz"].map(
    lambda value: f"{value:.3f}"
)
continuous_equivalence_display["Raw p"] = continuous_equivalence_display["TOST raw p"].map(format_p)
continuous_equivalence_display["Holm p (3 models)"] = continuous_equivalence_display["Holm p"].map(
    format_p
)
continuous_equivalence_display = continuous_equivalence_display[
    [
        "Condition",
        "Output",
        "Open-weight model",
        "n",
        "Mean difference [90% CI]",
        "Margin (raw, +/-)",
        "Cohen dz",
        "Raw p",
        "Holm p (3 models)",
        "Equivalent",
    ]
].reset_index(drop=True)

ordinal_equivalence_display = ordinal_equivalence.copy()
ordinal_equivalence_display["Rank-biserial r [90% CI]"] = ordinal_equivalence_display.apply(
    lambda row: f"{row['Rank-biserial r']:.3f} [{row['CI low (90%)']:.3f}, {row['CI high (90%)']:.3f}]",
    axis=1,
)
ordinal_equivalence_display["Margin (+/-)"] = ordinal_equivalence_display["Margin"].map(
    lambda value: f"{value:.2f}"
)
ordinal_equivalence_display["Raw p"] = ordinal_equivalence_display["TOST raw p"].map(format_p)
ordinal_equivalence_display["Holm p (3 models)"] = ordinal_equivalence_display["Holm p"].map(
    format_p
)
ordinal_equivalence_display["Lower / higher / tied"] = ordinal_equivalence_display.apply(
    lambda row: f"{row['Lower']} / {row['Higher']} / {row['Tied']}", axis=1
)
ordinal_equivalence_display = ordinal_equivalence_display[
    [
        "Condition",
        "Output",
        "Open-weight model",
        "n",
        "Rank-biserial r [90% CI]",
        "Margin (+/-)",
        "Raw p",
        "Holm p (3 models)",
        "Equivalent",
        "Lower / higher / tied",
    ]
].reset_index(drop=True)

print("Continuous MedCPT semantic similarity: paired TOST equivalence tests")
display(continuous_equivalence_display[continuous_equivalence_display["Output"] == "Process name"])
print("Ordinal LLM-judge scores: bootstrap TOST-style equivalence tests")
display(ordinal_equivalence_display[ordinal_equivalence_display["Output"] == "Process name"])

Continuous MedCPT semantic similarity: paired TOST equivalence tests


,Condition,Output,Open-weight model,n,Mean difference [90% CI],"Margin (raw, +/-)",Cohen dz,Raw p,Holm p (3 models),Equivalent
0,Full set,Process name,GPT-OSS:20B,120,"-0.0285 [-0.0450, -0.0119]",0.0546,-0.260,0.004917,0.009834,True
1,Full set,Process name,Gemma4:26B,120,"-0.0069 [-0.0227, 0.0088]",0.0519,-0.067,2.916e-06,8.748e-06,True
2,Full set,Process name,Mixtral:8x22B,120,"-0.0391 [-0.0559, -0.0223]",0.0556,-0.352,0.05343,0.05343,False
6,Pharmacologically actionable,Process name,GPT-OSS:20B,120,"-0.0201 [-0.0393, -0.0010]",0.0633,-0.159,0.0001454,0.0002908,True
7,Pharmacologically actionable,Process name,Gemma4:26B,120,"-0.0148 [-0.0329, 0.0032]",0.0597,-0.124,3.585e-05,0.0001075,True
8,Pharmacologically actionable,Process name,Mixtral:8x22B,120,"-0.0325 [-0.0523, -0.0127]",0.0655,-0.248,0.003344,0.003344,True


Ordinal LLM-judge scores: bootstrap TOST-style equivalence tests


,Condition,Output,Open-weight model,n,Rank-biserial r [90% CI],Margin (+/-),Raw p,Holm p (3 models),Equivalent,Lower / higher / tied
0,Full set,Process name,GPT-OSS:20B,120,"-0.317 [-0.550, -0.063]",0.50,0.1022,0.2043,False,29 / 17 / 74
1,Full set,Process name,Gemma4:26B,120,"-0.171 [-0.417, 0.097]",0.50,0.01345,0.04035,True,23 / 22 / 75
2,Full set,Process name,Mixtral:8x22B,120,"-0.616 [-0.789, -0.426]",0.50,0.8557,0.8557,False,37 / 11 / 72
6,Pharmacologically actionable,Process name,GPT-OSS:20B,120,"0.019 [-0.247, 0.297]",0.50,0.00235,0.00705,True,19 / 22 / 79
7,Pharmacologically actionable,Process name,Gemma4:26B,120,"-0.184 [-0.418, 0.059]",0.50,0.0132,0.0264,True,27 / 20 / 73
8,Pharmacologically actionable,Process name,Mixtral:8x22B,120,"-0.406 [-0.621, -0.168]",0.50,0.2469,0.2469,False,30 / 15 / 75


In [15]:
equivalence_counts = pd.concat(
    [
        continuous_equivalence.assign(Metric="MedCPT semantic similarity"),
        ordinal_equivalence.assign(Metric="LLM-judge score"),
    ],
    ignore_index=True,
)
equivalent_counts = (
    equivalence_counts.groupby(["Metric", "Condition", "Output"], sort=False)["Equivalent"]
    .sum()
    .rename("Equivalent open-weight models (of 3)")
    .reset_index()
)

print("TOST: count equivalent open-weight models")
display(equivalent_counts)

TOST: count equivalent open-weight models


,Metric,Condition,Output,Equivalent open-weight models (of 3)
0,MedCPT semantic similarity,Full set,Process name,2
1,MedCPT semantic similarity,Full set,Analytic narrative,1
2,MedCPT semantic similarity,Pharmacologically actionable,Process name,3
3,MedCPT semantic similarity,Pharmacologically actionable,Analytic narrative,1
4,LLM-judge score,Full set,Process name,1
5,LLM-judge score,Full set,Analytic narrative,0
6,LLM-judge score,Pharmacologically actionable,Process name,2
7,LLM-judge score,Pharmacologically actionable,Analytic narrative,0


### 10. Pathway incompleteness versus semantic similarity and LLM judge score: Spearman and partial correlation

Section 3's incompleteness-versus-size figure bins `missing_pct` into quartiles and plots mean +/- SEM semantic similarity (and stacked LLM judge score proportions) per bin, which is descriptive only. Here `missing_pct` (the fraction of druggable genes withheld from the `reduced_set`) is tested directly against both `semantic_similarity` and `llm_judge_score` for each LLM's `reduced_set` predictions using Spearman rank correlation; Spearman is used for both metrics rather than a parametric test since `llm_judge_score` is an ordinal 1-4 scale. Because gene-set size (`druggable_genes`) also varies across pathways and could confound an incompleteness effect, a partial Spearman correlation controlling for `druggable_genes` is reported alongside the raw correlation for each metric. Both correlations use the same paired bootstrap (20,000 resamples over the 120 pathways) for 95% CIs, and Holm adjustment is applied within each of the four raw-p families (one family per output x metric, across the four LLMs).

In [16]:
INCOMPLETENESS_METADATA_CSV = "Datasets/AlzKB/sampled_noise.csv"
INCOMPLETENESS_BOOTSTRAP_SEED = 20260727

incompleteness_metadata = (
    pd.read_csv(INCOMPLETENESS_METADATA_CSV)[["missing_pct", "druggable_genes"]]
    .reset_index()
    .rename(columns={"index": "pathway_id"})
)

INCOMPLETENESS_METRICS = {
    "MedCPT semantic similarity": "semantic_similarity",
    "LLM judge score": "llm_judge_score",
}


def spearman_rank_corr(x, y):
    """Spearman rho via Pearson correlation of ranks."""
    return np.corrcoef(rankdata(x), rankdata(y))[0, 1]


def partial_rank_corr(x, y, z):
    """Partial Spearman correlation of x and y controlling for z."""
    rho_xy = spearman_rank_corr(x, y)
    rho_xz = spearman_rank_corr(x, z)
    rho_yz = spearman_rank_corr(y, z)
    return (rho_xy - rho_xz * rho_yz) / np.sqrt((1 - rho_xz ** 2) * (1 - rho_yz ** 2))


def correlation_p_value(rho, n, control_variables=0):
    """Two-sided p-value for a (partial) correlation via its t-distribution approximation."""
    df = n - 2 - control_variables
    t_stat = rho * np.sqrt(df / (1 - rho ** 2))
    return 2 * t.sf(abs(t_stat), df)


def bootstrap_correlation_ci(missing_pct, metric_values, druggable_genes, resamples, seed):
    """Paired bootstrap 95% CIs for the raw and partial Spearman correlations."""
    rng = np.random.default_rng(seed)
    n = len(missing_pct)
    indices = rng.integers(0, n, size=(resamples, n))

    rank_missing = rankdata(missing_pct[indices], axis=1)
    rank_metric = rankdata(metric_values[indices], axis=1)
    rank_drug = rankdata(druggable_genes[indices], axis=1)

    def row_corr(a, b):
        a_centered = a - a.mean(axis=1, keepdims=True)
        b_centered = b - b.mean(axis=1, keepdims=True)
        return (a_centered * b_centered).sum(axis=1) / np.sqrt(
            (a_centered ** 2).sum(axis=1) * (b_centered ** 2).sum(axis=1)
        )

    boot_raw = row_corr(rank_missing, rank_metric)
    boot_xz = row_corr(rank_missing, rank_drug)
    boot_yz = row_corr(rank_metric, rank_drug)
    boot_partial = (boot_raw - boot_xz * boot_yz) / np.sqrt((1 - boot_xz ** 2) * (1 - boot_yz ** 2))

    raw_ci = np.quantile(boot_raw, [0.025, 0.975])
    partial_ci = np.quantile(boot_partial, [0.025, 0.975])
    return raw_ci, partial_ci


incompleteness_rows = []
for model_label in MODELS:
    for outcome_label in OUTCOMES:
        frame = frames[(model_label, outcome_label)]
        for metric_label, metric_column in INCOMPLETENESS_METRICS.items():
            reduced = frame.loc[
                frame["prediction_type"] == "reduced_set", ["pathway_id", metric_column]
            ]
            merged = reduced.merge(incompleteness_metadata, on="pathway_id", validate="one_to_one")
            assert len(merged) == EXPECTED_PATHWAYS

            missing_pct = merged["missing_pct"].to_numpy()
            metric_values = merged[metric_column].to_numpy()
            druggable_genes = merged["druggable_genes"].to_numpy()

            raw_rho = spearman_rank_corr(missing_pct, metric_values)
            partial_rho = partial_rank_corr(missing_pct, metric_values, druggable_genes)
            raw_ci, partial_ci = bootstrap_correlation_ci(
                missing_pct, metric_values, druggable_genes,
                BOOTSTRAP_RESAMPLES, INCOMPLETENESS_BOOTSTRAP_SEED,
            )

            incompleteness_rows.append(
                {
                    "Model": model_label,
                    "Output": outcome_label,
                    "Metric": metric_label,
                    "n": len(merged),
                    "Spearman rho": raw_rho,
                    "Spearman rho CI low": raw_ci[0],
                    "Spearman rho CI high": raw_ci[1],
                    "Spearman raw p": correlation_p_value(raw_rho, len(merged)),
                    "Partial rho (control: gene set size)": partial_rho,
                    "Partial rho CI low": partial_ci[0],
                    "Partial rho CI high": partial_ci[1],
                    "Partial raw p": correlation_p_value(partial_rho, len(merged), control_variables=1),
                }
            )

incompleteness_results = pd.DataFrame(incompleteness_rows)


def apply_incompleteness_holm_families(results):
    """Apply Holm adjustment within each output x metric family, across the four LLMs, separately for the raw and partial correlations."""
    results = results.copy()
    results["Spearman Holm p"] = np.nan
    results["Partial Holm p"] = np.nan
    for outcome_label in OUTCOMES:
        for metric_label in INCOMPLETENESS_METRICS:
            family_mask = results["Output"].eq(outcome_label) & results["Metric"].eq(metric_label)
            assert family_mask.sum() == len(MODELS)
            results.loc[family_mask, "Spearman Holm p"] = holm_adjust(
                results.loc[family_mask, "Spearman raw p"].to_numpy()
            )
            results.loc[family_mask, "Partial Holm p"] = holm_adjust(
                results.loc[family_mask, "Partial raw p"].to_numpy()
            )
    return results


incompleteness_results = apply_incompleteness_holm_families(incompleteness_results)

assert len(incompleteness_results) == len(MODELS) * len(OUTCOMES) * len(INCOMPLETENESS_METRICS)
assert incompleteness_results["n"].eq(EXPECTED_PATHWAYS).all()
assert (incompleteness_results["Spearman Holm p"] + 1e-15 >= incompleteness_results["Spearman raw p"]).all()
assert (incompleteness_results["Partial Holm p"] + 1e-15 >= incompleteness_results["Partial raw p"]).all()

incompleteness_results

,Model,Output,Metric,n,Spearman rho,Spearman rho CI low,Spearman rho CI high,Spearman raw p,Partial rho (control: gene set size),Partial rho CI low,Partial rho CI high,Partial raw p,Spearman Holm p,Partial Holm p
0,GPT-5.4 mini,Process name,MedCPT semantic similarity,120,0.084687,-0.113807,0.279285,0.357758,0.053952,-0.149267,0.254575,0.560053,0.807353,1.000000
1,GPT-5.4 mini,Process name,LLM judge score,120,0.095488,-0.094640,0.276775,0.299531,0.014786,-0.164124,0.192079,0.873191,1.000000,1.000000
2,GPT-5.4 mini,Analytic narrative,MedCPT semantic similarity,120,0.003511,-0.181274,0.187644,0.969640,0.020103,-0.159800,0.198937,0.828205,1.000000,1.000000
3,GPT-5.4 mini,Analytic narrative,LLM judge score,120,-0.068951,-0.272273,0.141316,0.454278,-0.088801,-0.270548,0.099997,0.336865,1.000000,0.773271
4,GPT-OSS:20B,Process name,MedCPT semantic similarity,120,0.117548,-0.070678,0.297868,0.201026,0.034878,-0.162161,0.222163,0.706491,0.804105,1.000000
5,GPT-OSS:20B,Process name,LLM judge score,120,0.097498,-0.078697,0.264841,0.289426,-0.044832,-0.229276,0.138510,0.628289,1.000000,1.000000
6,GPT-OSS:20B,Analytic narrative,MedCPT semantic similarity,120,0.100611,-0.086821,0.278270,0.274229,0.018828,-0.163321,0.196692,0.838952,0.998708,1.000000
7,GPT-OSS:20B,Analytic narrative,LLM judge score,120,-0.046969,-0.233620,0.136096,0.610460,-0.059748,-0.221607,0.105287,0.518622,1.000000,0.773271
8,Gemma4:26B,Process name,MedCPT semantic similarity,120,-0.048975,-0.227393,0.134523,0.595281,-0.132006,-0.318292,0.060944,0.152405,0.807353,0.609619
9,Gemma4:26B,Process name,LLM judge score,120,0.065184,-0.121690,0.243060,0.479357,-0.103344,-0.296839,0.096533,0.263376,1.000000,0.790129


In [17]:
def format_p(value):
    return f"{value:.4g}"


incompleteness_display = incompleteness_results.copy()
incompleteness_display["Spearman rho [95% CI]"] = incompleteness_display.apply(
    lambda row: f"{row['Spearman rho']:.3f} [{row['Spearman rho CI low']:.3f}, {row['Spearman rho CI high']:.3f}]",
    axis=1,
)
incompleteness_display["Partial rho [95% CI]"] = incompleteness_display.apply(
    lambda row: (
        f"{row['Partial rho (control: gene set size)']:.3f} "
        f"[{row['Partial rho CI low']:.3f}, {row['Partial rho CI high']:.3f}]"
    ),
    axis=1,
)
incompleteness_display["Spearman Holm p (4 LLMs)"] = incompleteness_display["Spearman Holm p"].map(format_p)
incompleteness_display["Partial Holm p (4 LLMs)"] = incompleteness_display["Partial Holm p"].map(format_p)
incompleteness_display = incompleteness_display[
    [
        "Model",
        "Output",
        "Metric",
        "n",
        "Spearman rho [95% CI]",
        "Spearman Holm p (4 LLMs)",
        "Partial rho [95% CI]",
        "Partial Holm p (4 LLMs)",
    ]
].reset_index(drop=True)

for metric_label in INCOMPLETENESS_METRICS:
    print(f"{metric_label} vs pathway incompleteness (missing_pct): Spearman and partial correlation controlling for druggable gene-set size")
    for outcome_label in OUTCOMES:
        display(
            incompleteness_display[
                (incompleteness_display["Metric"] == metric_label)
                & (incompleteness_display["Output"] == outcome_label)
            ]
        )

MedCPT semantic similarity vs pathway incompleteness (missing_pct): Spearman and partial correlation controlling for druggable gene-set size


,Model,Output,Metric,n,Spearman rho [95% CI],Spearman Holm p (4 LLMs),Partial rho [95% CI],Partial Holm p (4 LLMs)
0,GPT-5.4 mini,Process name,MedCPT semantic similarity,120,"0.085 [-0.114, 0.279]",0.8074,"0.054 [-0.149, 0.255]",1
4,GPT-OSS:20B,Process name,MedCPT semantic similarity,120,"0.118 [-0.071, 0.298]",0.8041,"0.035 [-0.162, 0.222]",1
8,Gemma4:26B,Process name,MedCPT semantic similarity,120,"-0.049 [-0.227, 0.135]",0.8074,"-0.132 [-0.318, 0.061]",0.6096
12,Mixtral:8x22B,Process name,MedCPT semantic similarity,120,"0.102 [-0.074, 0.277]",0.8074,"0.008 [-0.187, 0.206]",1


,Model,Output,Metric,n,Spearman rho [95% CI],Spearman Holm p (4 LLMs),Partial rho [95% CI],Partial Holm p (4 LLMs)
2,GPT-5.4 mini,Analytic narrative,MedCPT semantic similarity,120,"0.004 [-0.181, 0.188]",1,"0.020 [-0.160, 0.199]",1
6,GPT-OSS:20B,Analytic narrative,MedCPT semantic similarity,120,"0.101 [-0.087, 0.278]",0.9987,"0.019 [-0.163, 0.197]",1
10,Gemma4:26B,Analytic narrative,MedCPT semantic similarity,120,"-0.007 [-0.199, 0.178]",1,"-0.009 [-0.187, 0.165]",1
14,Mixtral:8x22B,Analytic narrative,MedCPT semantic similarity,120,"0.106 [-0.081, 0.286]",0.9987,"0.026 [-0.160, 0.209]",1


LLM judge score vs pathway incompleteness (missing_pct): Spearman and partial correlation controlling for druggable gene-set size


,Model,Output,Metric,n,Spearman rho [95% CI],Spearman Holm p (4 LLMs),Partial rho [95% CI],Partial Holm p (4 LLMs)
1,GPT-5.4 mini,Process name,LLM judge score,120,"0.095 [-0.095, 0.277]",1,"0.015 [-0.164, 0.192]",1
5,GPT-OSS:20B,Process name,LLM judge score,120,"0.097 [-0.079, 0.265]",1,"-0.045 [-0.229, 0.139]",1
9,Gemma4:26B,Process name,LLM judge score,120,"0.065 [-0.122, 0.243]",1,"-0.103 [-0.297, 0.097]",0.7901
13,Mixtral:8x22B,Process name,LLM judge score,120,"0.042 [-0.151, 0.233]",1,"-0.207 [-0.380, -0.017]",0.09557


,Model,Output,Metric,n,Spearman rho [95% CI],Spearman Holm p (4 LLMs),Partial rho [95% CI],Partial Holm p (4 LLMs)
3,GPT-5.4 mini,Analytic narrative,LLM judge score,120,"-0.069 [-0.272, 0.141]",1,"-0.089 [-0.271, 0.100]",0.7733
7,GPT-OSS:20B,Analytic narrative,LLM judge score,120,"-0.047 [-0.234, 0.136]",1,"-0.060 [-0.222, 0.105]",0.7733
11,Gemma4:26B,Analytic narrative,LLM judge score,120,"0.012 [-0.191, 0.210]",1,"-0.120 [-0.301, 0.068]",0.7733
15,Mixtral:8x22B,Analytic narrative,LLM judge score,120,"0.023 [-0.166, 0.216]",1,"-0.109 [-0.257, 0.049]",0.7733
